# DiMMAD anomaly scoring on ZTF/RELAISS features

1. Loads a RELAISS feature table (with durations + sky coords)
2. Applies basic quality cuts (duration, Galactic latitude)
3. Cross-matches to ALeRCE explorer export to get class labels
4. Trains DiMMAD (distclassipy DistanceAnomaly) on *known* classes
5. Scores an unlabeled/unknown set and ranks anomalies




In [1]:
# --- Imports & settings ---
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

from astropy.coordinates import SkyCoord
import astropy.units as u

from distclassipy.anomaly import DistanceAnomaly

import matplotlib.pyplot as plt

from relaiss import constants
import relaiss as rl

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)


In [2]:
# --- Paths ---
DATA_DIR = Path("data")          
OUT_DIR  = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RELAISS_CSV = DATA_DIR / "reference_20k_with_durations.csv"
ALERCE_EXPLORER_CSV = DATA_DIR / "btsztf.csv"
#ALERCE_EXPLORER_CSV = Path("experiments") / "explorer.csv"  

RELAISS_CSV, ALERCE_EXPLORER_CSV


(PosixPath('data/reference_20k_with_durations.csv'),
 PosixPath('data/btsztf.csv'))

## Load data

In [5]:
df = pd.read_csv(RELAISS_CSV, low_memory=False)
df.head()

,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,rExtNSigma,iKronRad,yExtNSigma,ymomentYY,ny,zstackDetectID,gEpoch,iFPSFFlux,nStackDetections,KronMag,ypsfLikelihood,iinfoFlag,zFPSFFlux,zpsfCore,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days
0,59850.515694,18.058201,19.991794,6.032847,12.011400,18.044248,0.521299,0.225308,0.0,17.598600,0.000000,NaN,33.467286,31.981111,1.021000,-0.492451,0.083333,0.109230,0.027208,0.016575,1,NaN,NaN,NaN,NaN,2,12.119703,0.211326,0.249921,20.899493,0.000295,0.000144,0.071492,0.001672,-0.007540,-0.002478,NaN,-0.005436,False,0.0,0.058249,0.903369,0.903369,0.903369,0.000000,0.074867,0.576231,6.117006e-02,0.170234,0.000000,NaN,16.194011,4.325329,0.154256,0.477562,0.045843,0.051900,0.119613,0.008798,0.615587,4.181948,0.153224,0.045004,13.055632,0.638666,3.519174,0.126541,0.123748,7.833686,0.000581,0.000123,0.054051,0.000855,0.021606,0.012067,NaN,0.006085,0.0,ZTF17aabtvsy,158.883762,37.649729,5.0,19226.0,panstarrs,dr2,6.246375,1.341795e-15,1.341795e-15,"{'iinfoFlag3': 196608.0, 'zpsfQfPerfect': 0.99...",1.531816e+17,NaN,0.987186,158.883575,37.649555,0.077098,0.019274,-22.734511,0.749100,r,0.829189,...,13.785300,17.996799,23.644100,0.228099,13.0,2.523244e+18,55923.256190,0.000530,5.0,14.982000,0.000000,262160391.0,0.000617,0.425564,-0.002408,0.943950,16.888700,0.001657,0.001063,0.001878

## Basic cuts
- remove long-duration objects (duration_days > 200)
- avoid Galactic plane (|b| < 15°)

In [7]:
DURATION_MAX_DAYS = 200
LAT_CUT_DEG = 15.0

coords = SkyCoord(ra=df["ra"].values * u.deg, dec=df["dec"].values * u.deg, frame="icrs")
b = coords.galactic.b.deg

mask = (df["duration_days"] <= DURATION_MAX_DAYS) & (np.abs(b) >= LAT_CUT_DEG)
df_cut = df.loc[mask].copy()

print(f"Kept {len(df_cut):,} / {len(df):,} rows")
df_cut[["ra","dec","duration_days"]].describe()

df_cut.head()

Kept 17,254 / 25,515 rows


,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,rExtNSigma,iKronRad,yExtNSigma,ymomentYY,ny,zstackDetectID,gEpoch,iFPSFFlux,nStackDetections,KronMag,ypsfLikelihood,iinfoFlag,zFPSFFlux,zpsfCore,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days
0,59850.515694,18.058201,19.991794,6.032847,12.011400,18.044248,0.521299,0.225308,0.0,17.598600,0.000000,NaN,33.467286,31.981111,1.021000,-0.492451,0.083333,0.109230,0.027208,0.016575,1,NaN,NaN,NaN,NaN,2,12.119703,0.211326,0.249921,20.899493,0.000295,0.000144,0.071492,0.001672,-0.007540,-0.002478,NaN,-0.005436,False,0.0,0.058249,0.903369,0.903369,0.903369,0.000000,0.074867,0.576231,6.117006e-02,0.170234,0.000000,NaN,16.194011,4.325329,0.154256,0.477562,0.045843,0.051900,0.119613,0.008798,0.615587,4.181948,0.153224,0.045004,13.055632,0.638666,3.519174,0.126541,0.123748,7.833686,0.000581,0.000123,0.054051,0.000855,0.021606,0.012067,NaN,0.006085,0.0,ZTF17aabtvsy,158.883762,37.649729,5.0,19226.0,panstarrs,dr2,6.246375,1.341795e-15,1.341795e-15,"{'iinfoFlag3': 196608.0, 'zpsfQfPerfect': 0.99...",1.531816e+17,NaN,0.987186,158.883575,37.649555,0.077098,0.019274,-22.734511,0.749100,r,0.829189,...,13.785300,17.996799,23.644100,0.228099,13.0,2.523244e+18,55923.256190,0.000530,5.0,14.982000,0.000000,262160391.0,0.000617,0.425564,-0.002408,0.943950,16.888700,0.001657,0.001063,0.001878

## Cross-match to labeled catalog
Original notebook tried to match by `ZTFID` and now switched to coordinate matching.

Instead:
- converts ALeRCE RA/Dec strings into degrees
- matches each `df_cut` row to nearest ALeRCE object
- keeps matches within 1 arcsec


In [9]:
zdf = pd.read_csv(ALERCE_EXPLORER_CSV, low_memory=False)

df_cut["ZTFID"].describe()

df_copycut = df_cut.copy()

In [11]:
df_matched = df_copycut.merge(
    zdf,
    on="ZTFID",
    how="inner"   
)

len(set(df_copycut["ZTFID"]) & set(zdf["ZTFID"]))

3831

In [13]:
tmp = df_copycut.merge(
    zdf,
    on="ZTFID",
    how="left",
    indicator=True
)

df_matched   = tmp[tmp["_merge"] == "both"].drop(columns=["_merge"])

df_matched.head()

,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days,IAUID,RA,Dec,peakt,peakfilt,peakmag,peakabs,duration,rise,fade,type,redshift,b_y,A_V
0,59850.515694,18.058201,19.991794,6.032847,12.011400,18.044248,0.521299,0.225308,0.000000,17.598600,0.000000,NaN,33.467286,31.981111,1.021000,-0.492451,0.083333,0.109230,0.027208,0.016575,1,NaN,NaN,NaN,NaN,2,12.119703,0.211326,0.249921,20.899493,0.000295,0.000144,0.071492,0.001672,-0.007540,-0.002478,NaN,-0.005436,False,0.0,0.058249,0.903369,0.903369,0.903369,0.000000,0.074867,0.576231,0.061170,0.170234,0.000000,NaN,16.194011,4.325329,0.154256,0.477562,0.045843,0.051900,0.119613,0.008798,0.615587,4.181948,0.153224,0.045004,13.055632,0.638666,3.519174,0.126541,0.123748,7.833686,0.000581,0.000123,0.054051,0.000855,0.021606,0.012067,NaN,0.006085,0.000000,ZTF17aabtvsy,158.883762,37.649729,5.0,19226.0,panstarrs,dr2,6.246375,1.341795e-15,1.341795e-15,"{'iinfoFlag3': 196608.0, 'zpsfQfPerfect': 0.99...",1.531816e+17,NaN,0.987186,158.883575,37.649555,0.077098,0.019274,-22.734511,0.749100,r,0.829189,...,-0.002408,0.943950,16.888700,0.001657,0.001063,0.001878,6.914810e-07,262160391.0,0.927463,16.0,1.17831,0.319913,4.597210e-06,12.0,0.522189,158.883637,0.004043,3.738386e+15,NaN,17.254601,2.25,0.001,4.69666,1.093620e-06,262160391.0,6.85620,24.

In [15]:
df_unmatched = tmp[tmp["_merge"] == "left_only"].drop(columns=["_merge"])

df_unmatched.head()

,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days,IAUID,RA,Dec,peakt,peakfilt,peakmag,peakabs,duration,rise,fade,type,redshift,b_y,A_V
2,59861.468773,18.025000,0.000000,NaN,1.246246,5.971481,1.945101,0.815205,0.1,18.274099,0.021192,NaN,1.415106,9.945428,1.760750,0.595434,0.090909,-0.098382,-0.249100,0.013544,1,NaN,NaN,NaN,NaN,2,5.954756,0.602849,0.198726,31.826992,0.014541,0.004381,0.019289,0.004223,NaN,0.076705,NaN,0.087738,False,0.0,0.042273,0.000000,NaN,0.240076,1.968606,0.103605,0.153046,1.423831e-17,0.051127,1.643622,0.000000,1.198672,1.987739,0.178663,0.222787,0.020328,0.044112,0.063622,0.014004,0.489360,1.540522,0.171408,0.075891,7.011332,0.695852,1.965831,0.294856,0.083630,10.050958,0.005109,0.001060,0.010062,0.000792,NaN,0.070404,0.089636,0.111978,0.000000,ZTF17aaaycpc,105.863596,32.068390,2.0,10296.0,panstarrs,dr2,8.549133,7.403929e-15,7.403929e-15,"{'raStackErr': 0.0010000000474974, 'cz': 0.530...",1.464811e+17,NaN,0.998975,105.864122,32.068686,0.516699,0.259615,-20.990470,1.067820,r,1.929947,...,-0.014049,0.818664,20.723400,0.037106,0.000009,0.055614,7.877810e-07,35651585.0,1.039530,4.0,1.47531,NaN,9.757030e-07,0.0,0.229726,105.864119,0.000017,3.792412e+15,NaN,21.314301,2.50000,0.001,NaN,1.076550e-07,35651585.0,0.858702,24.983700,0.000006,0.006697,0.0,

In [17]:
alerce_coords = SkyCoord(
    ra=zdf["RA"].astype(str),
    dec=zdf["Dec"].astype(str),
    unit=(u.hourangle, u.deg)
)
zdf = zdf.copy()
zdf["ra_deg"] = alerce_coords.ra.deg
zdf["dec_deg"] = alerce_coords.dec.deg

In [19]:
ref_coords = SkyCoord(ra=df_cut["ra"].values * u.deg, dec=df_cut["dec"].values * u.deg)

idx, sep2d, _ = ref_coords.match_to_catalog_sky(alerce_coords)
mask_match = sep2d < (1.0 * u.arcsec)

In [21]:
df_copycut["alerce_idx"] = idx
df_copycut["matched"] = mask_match

df_copycut.head()

,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,yExtNSigma,ymomentYY,ny,zstackDetectID,gEpoch,iFPSFFlux,nStackDetections,KronMag,ypsfLikelihood,iinfoFlag,zFPSFFlux,zpsfCore,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days,alerce_idx,matched
0,59850.515694,18.058201,19.991794,6.032847,12.011400,18.044248,0.521299,0.225308,0.0,17.598600,0.000000,NaN,33.467286,31.981111,1.021000,-0.492451,0.083333,0.109230,0.027208,0.016575,1,NaN,NaN,NaN,NaN,2,12.119703,0.211326,0.249921,20.899493,0.000295,0.000144,0.071492,0.001672,-0.007540,-0.002478,NaN,-0.005436,False,0.0,0.058249,0.903369,0.903369,0.903369,0.000000,0.074867,0.576231,6.117006e-02,0.170234,0.000000,NaN,16.194011,4.325329,0.154256,0.477562,0.045843,0.051900,0.119613,0.008798,0.615587,4.181948,0.153224,0.045004,13.055632,0.638666,3.519174,0.126541,0.123748,7.833686,0.000581,0.000123,0.054051,0.000855,0.021606,0.012067,NaN,0.006085,0.0,ZTF17aabtvsy,158.883762,37.649729,5.0,19226.0,panstarrs,dr2,6.246375,1.341795e-15,1.341795e-15,"{'iinfoFlag3': 196608.0, 'zpsfQfPerfect': 0.99...",1.531816e+17,NaN,0.987186,158.883575,37.649555,0.077098,0.019274,-22.734511,0.749100,r,0.829189,...,23.644100,0.228099,13.0,2.523244e+18,55923.256190,0.000530,5.0,14.982000,0.000000,262160391.0,0.000617,0.425564,-0.002408,0.943950,16.888700,0.001657,0.001063,0.001878,6.914810e-07,2621603

In [23]:
df_matched = df_copycut.loc[mask_match].merge(
    zdf[["ra_deg","dec_deg","ZTFID","type","IAUID"]],
    left_on="alerce_idx",
    right_index=True,
    how="left"
)

df_unmatched = df_copycut.loc[~mask_match].copy()

df_matched.head()
df_unmatched.head()


,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,yExtNSigma,ymomentYY,ny,zstackDetectID,gEpoch,iFPSFFlux,nStackDetections,KronMag,ypsfLikelihood,iinfoFlag,zFPSFFlux,zpsfCore,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days,alerce_idx,matched
2,59861.468773,18.025000,0.000000,NaN,1.246246,5.971481,1.945101,0.815205,0.1,18.274099,0.021192,NaN,1.415106,9.945428,1.760750,0.595434,0.090909,-0.098382,-0.249100,0.013544,1,NaN,NaN,NaN,NaN,2,5.954756,0.602849,0.198726,31.826992,0.014541,0.004381,0.019289,0.004223,NaN,0.076705,NaN,0.087738,False,0.0,0.042273,0.000000,NaN,0.240076,1.968606,0.103605,0.153046,1.423831e-17,0.051127,1.643622,0.000000,1.198672,1.987739,0.178663,0.222787,0.020328,0.044112,0.063622,0.014004,0.489360,1.540522,0.171408,0.075891,7.011332,0.695852,1.965831,0.294856,0.083630,10.050958,0.005109,0.001060,0.010062,0.000792,NaN,0.070404,0.089636,0.111978,0.000000,ZTF17aaaycpc,105.863596,32.068390,2.0,10296.0,panstarrs,dr2,8.549133,7.403929e-15,7.403929e-15,"{'raStackErr': 0.0010000000474974, 'cz': 0.530...",1.464811e+17,NaN,0.998975,105.864122,32.068686,0.516699,0.259615,-20.990470,1.067820,r,1.929947,...,0.691058,0.117420,0.0,2.523214e+18,56301.695611,0.000011,4.0,21.356400,0.489529,35651585.0,0.000019,0.726127,-0.014049,0.818664,20.723400,0.037106,0.000009,0.055614,7.877810e-07,35651585.0,1.039530,4.0,

In [25]:
print(f"Matched: {len(df_matched):,}")
print(f"Unmatched: {len(df_unmatched):,}")

Matched: 3,835
Unmatched: 13,419


In [27]:
USE_HOST = False 

# Grab our RELAISS features
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

matchlc_cols   = overlap(default_lc_features, df_matched)
matchhost_cols = overlap(default_host_features, df_matched) if USE_HOST else []
matchfeature_cols = matchlc_cols + matchhost_cols

print(f"Found {len(matchlc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(matchhost_cols)} host features in filtered data.")
print(f"Total candidate features: {len(matchfeature_cols)}")

Found 25 LC features in filtered data.
Total candidate features: 25


In [195]:
unmatchlc_cols   = overlap(default_lc_features, df_unmatched)
unmatchhost_cols = overlap(default_host_features, df_unmatched) if USE_HOST else []
unmatchfeature_cols = unmatchlc_cols + unmatchhost_cols

print(f"Found {len(unmatchlc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(unmatchhost_cols)} host features in filtered data.")
print(f"Total candidate features: {len(unmatchfeature_cols)}")

Found 25 LC features in filtered data.
Total candidate features: 25


In [29]:
zdf = pd.read_csv(ALERCE_EXPLORER_CSV, low_memory=False)

df_cut["ZTFID"].describe()

df_copycut = df_cut.copy()

df_matched = df_copycut.merge(
    zdf,
    on="ZTFID",
    how="inner"   
)

len(set(df_copycut["ZTFID"]) & set(zdf["ZTFID"]))

#df_cut["ZTFID"].isna().sum()
#df_cut["ZTFID"].duplicated().sum()

tmp = df_copycut.merge(
    zdf,
    on="ZTFID",
    how="left",
    indicator=True
)

df_matched   = tmp[tmp["_merge"] == "both"].drop(columns=["_merge"])
df_unmatched = tmp[tmp["_merge"] == "left_only"].drop(columns=["_merge"])


In [31]:
zdf = pd.read_csv(ALERCE_EXPLORER_CSV, low_memory=False)

# ALeRCE explorer export often stores RA as hourangle string and Dec as deg string
alerce_coords = SkyCoord(
    ra=zdf["RA"].astype(str),
    dec=zdf["Dec"].astype(str),
    unit=(u.hourangle, u.deg)
)
zdf = zdf.copy()
zdf["ra_deg"] = alerce_coords.ra.deg
zdf["dec_deg"] = alerce_coords.dec.deg

ref_coords = SkyCoord(ra=df_cut["ra"].values * u.deg, dec=df_cut["dec"].values * u.deg)

idx, sep2d, _ = ref_coords.match_to_catalog_sky(alerce_coords)
mask_match = sep2d < (1.0 * u.arcsec)

df_cut = df_cut.copy()
df_cut["alerce_idx"] = idx
df_cut["matched"] = mask_match

df_matched = df_cut.loc[mask_match].merge(
    zdf[["ra_deg","dec_deg","ZTFID","type","IAUID"]],
    left_on="alerce_idx",
    right_index=True,
    how="left"
)

df_unmatched = df_cut.loc[~mask_match].copy()

print(f"Matched: {len(df_matched):,}")
print(f"Unmatched: {len(df_unmatched):,}")

# After df_matched is created:
print([c for c in df_matched.columns if "ZTF" in c.upper()])

# Common fix:
if "ZTFID_x" in df_matched.columns:
    df_matched = df_matched.rename(columns={"ZTFID_x": "ZTFID"})
elif "ZTFID_y" in df_matched.columns:
    df_matched = df_matched.rename(columns={"ZTFID_y": "ZTFID"})

df_matched[["ZTFID","type","IAUID","ra","dec"]].head()


Matched: 3,835
Unmatched: 13,419
['ZTFID_x', 'ZTFID_y']


,ZTFID,type,IAUID,ra,dec
0,ZTF17aabtvsy,SN Ia,SN2022yei,158.883762,37.649729
1,ZTF17aabvong,SN Ia,SN2024xxq,31.282032,11.248651
23,ZTF18aacnlxz,SN II,SN2020aavr,134.954664,38.109070
27,ZTF18aaczmob,-,-,105.519506,49.059275
28,ZTF18aadaexi,-,AT2021dpr,125.241560,36.312320


## Map labels to classifications
Training set of present classes and an 'unknown/unlabeled' set to score.


In [33]:
TYPE_MAP = {
    # Superluminous
    "SLSN-I": "SLSN",
    "SLSN-II": "SLSN",

    # Type II
    "SN II": "II",
    "SN IIP": "II",

    # Type IIb / IIn
    "SN IIb": "IIb",
    "SN IIn": "IIn",

    # Thermonuclear (Ia family)
    "SN Ia": "Ia",
    "SN Ia-91T": "Ia",
    "SN Ia-91bg": "Ia",
    "SN Ia-SC": "Ia",
    "SN Iax": "Ia",

    # Stripped-envelope
    "SN Ib": "Ibc",
    "SN Ib/c": "Ibc",
    "SN Ic": "Ibc",
    "SN Ic-BL": "Ibc",

    # TDE
    "TDE": "TDE",
    "TDE-He": "TDE",
}

def map_type(t):
    if pd.isna(t):
        return np.nan
    t = str(t).strip()
    return TYPE_MAP.get(t, "UNKNOWN")

In [35]:
df_matched["type_group"] = df_matched["type"].apply(map_type)
print(df_matched["type_group"].value_counts(dropna=False))

known_groups = ["SLSN", "II", "IIb", "IIn", "Ia", "Ibc", "TDE"]

df_train = df_matched[df_matched["type_group"].isin(known_groups)].copy()

df_anom = pd.concat(
    [df_unmatched, df_matched[df_matched["type_group"] == "UNKNOWN"]],
    ignore_index=True
)

print(f"Train rows: {len(df_train):,}")
print(f"To-score rows: {len(df_anom):,}")

type_group
Ia         2116
UNKNOWN    1143
II          320
Ibc         125
IIn          49
IIb          41
SLSN         24
TDE          17
Name: count, dtype: int64
Train rows: 2,692
To-score rows: 14,562


## Feature selection + preprocessing

- Numeric columns are used (avoid `ZTFID` strings)
- handle ±inf → NaN
- KNN impute, then standardise
- Use ReLaiss features


In [37]:
def preprocess(train_df: pd.DataFrame, score_df: pd.DataFrame, matchfeature_cols, n_neighbors=5):
    X_train_raw = train_df[matchfeature_cols].replace([np.inf, -np.inf], np.nan)
    X_score_raw = score_df[matchfeature_cols].replace([np.inf, -np.inf], np.nan)

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="distance")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_score_imp = imputer.transform(X_score_raw)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_imp)
    X_score = scaler.transform(X_score_imp)

    return X_train, X_score, imputer, scaler

X_train, X_score, imputer, scaler = preprocess(df_train, df_anom, matchfeature_cols)
y_train = df_train["type_group"].values

X_train.shape, X_score.shape


((2692, 25), (14562, 25))

## DiMMAD scoring (median-median)
`cluster_agg` controls how each class cluster is aggregated; `metric_agg` controls aggregation across distance metrics.


In [39]:
dimmad_medmed = DistanceAnomaly(
    cluster_agg="median",
    metric_agg="median",
    normalize_scores=True
)

dimmad_medmed.fit(X_train, y_train)
scores_medmed = dimmad_medmed.score_samples(X_score)

df_anom = df_anom.copy()
df_anom["dimmad_medmed"] = scores_medmed

df_anom[["dimmad_medmed"]].describe()
df_anom.head()


,t0,g_peak_mag,g_peak_time,g_rise_time,g_decline_time,g_duration_above_half_flux,g_amplitude,g_skewness,g_beyond_2sigma,r_peak_mag,r_peak_time,r_rise_time,r_decline_time,r_duration_above_half_flux,r_amplitude,r_skewness,r_beyond_2sigma,mean_g-r,g-r_at_g_peak,mean_color_rate,g_n_peaks,g_dt_main_to_secondary_peak,g_dmag_secondary_peak,g_secondary_peak_prominence,g_secondary_peak_width,r_n_peaks,r_dt_main_to_secondary_peak,r_dmag_secondary_peak,r_secondary_peak_prominence,r_secondary_peak_width,g_max_rolling_variance,g_mean_rolling_variance,r_max_rolling_variance,r_mean_rolling_variance,g_rise_local_curvature,g_decline_local_curvature,r_rise_local_curvature,r_decline_local_curvature,features_valid,t0_err,g_peak_mag_err,g_peak_time_err,g_rise_time_err,g_decline_time_err,g_duration_above_half_flux_err,g_amplitude_err,g_skewness_err,g_beyond_2sigma_err,r_peak_mag_err,r_peak_time_err,r_rise_time_err,r_decline_time_err,r_duration_above_half_flux_err,r_amplitude_err,r_skewness_err,r_beyond_2sigma_err,mean_g-r_err,g-r_at_g_peak_err,mean_color_rate_err,g_n_peaks_err,g_dt_main_to_secondary_peak_err,g_dmag_secondary_peak_err,g_secondary_peak_prominence_err,g_secondary_peak_width_err,r_n_peaks_err,r_dt_main_to_secondary_peak_err,r_dmag_secondary_peak_err,r_secondary_peak_prominence_err,r_secondary_peak_width_err,g_max_rolling_variance_err,g_mean_rolling_variance_err,r_max_rolling_variance_err,r_mean_rolling_variance_err,g_rise_local_curvature_err,g_decline_local_curvature_err,r_rise_local_curvature_err,r_decline_local_curvature_err,features_valid_err,ZTFID,ra,dec,Unnamed: 0,idx,best_cat,best_cat_release,query_time,smallcone_posterior,missedcat_posterior,extra_cat_cols,host_objID,host_name,host_total_posterior,host_ra,host_dec,host_redshift_mean,host_redshift_std,host_absmag_mean,host_absmag_std,host_absmag_info,host_offset_mean,...,KronMag,ypsfLikelihood,iinfoFlag,zFPSFFlux,zpsfCore,momentXY,ypsfMinorFWHM,zPSFMag,yraErr,iFApFlux,zPSFMagErr,iPSFFluxErr,rinfoFlag,ipsfMinorFWHM,inFrames,gpsfMinorFWHM,gmomentXX,iKronFluxErr,ng,rpsfCore,raStack,zFKronFlux,uniquePspsSTid,objAltName1,iPSFMag,zApRadius,decStackErr,gmomentR1,yApFluxErr,zinfoFlag,rmomentR1,gzp,iKronFlux,randomID,surveyID,cx,imomentRH,ydec,gyPosErr,rra,rKronRad,zpsfMinorFWHM,yApFillFac,yKronMag,uniquePspsOBid,rpsfTheta,zApMagErr,iippDetectID,iinfoFlag2,ystackDetectID,batchID,rFmeanflxR5,zFmeanflxR7,iEpoch,zxPosErr,rpsfLikelihood,yPSFFlux,yEpoch,rKronMag,gdecErr,zmomentXX,yinfoFlag,zpsfChiSq,zexpTime,zKronMagErr,rKronFluxErr,gKronFluxErr,gKronFlux,rFApFlux,gPSFFlux,gKronMag,iraErr,gstackImageID,yFmeanflxR6,graErr,ipsfQf,iFmeanflxR5,zExtNSigma,ypsfCore,zFmeanflxR6,iFmeanflxR6,mjd_extracted,tns_redshift,latest_magnitude,oldest_alert,newest_alert,peak_phase,antares_oldest_alert,antares_newest_alert,antares_duration,duration_days,alerce_idx,matched,ra_deg,dec_deg,ZTFID_y,type,IAUID,type_group,dimmad_medmed
0,59861.468773,18.025000,0.000000,NaN,1.246246,5.971481,1.945101,0.815205,0.1,18.274099,0.021192,NaN,1.415106,9.945428,1.760750,0.595434,0.090909,-0.098382,-0.249100,0.013544,1,NaN,NaN,NaN,NaN,2,5.954756,0.602849,0.198726,31.826992,0.014541,0.004381,0.019289,0.004223,NaN,0.076705,NaN,0.087738,False,0.0,0.042273,0.000000,NaN,0.240076,1.968606,0.103605,0.153046,1.423831e-17,0.051127,1.643622,0.000000,1.198672,1.987739,0.178663,0.222787,0.020328,0.044112,0.063622,0.014004,0.489360,1.540522,0.171408,0.075891,7.011332,0.695852,1.965831,0.294856,0.083630,10.050958,0.005109,0.001060,0.010062,0.000792,NaN,0.070404,0.089636,0.111978,0.000000,ZTF17aaaycpc,105.863596,32.068390,2.0,10296.0,panstarrs,dr2,8.549133,7.403929e-15,7.403929e-15,"{'raStackErr': 0.0010000000474974, 'cz': 0.530...",1.464811e+17,NaN,0.998975,105.864122,32.068686,0.516699,0.259615,-20.990470,1.067820,r,1.929947,...,21.356400,0.489529,35651585.0,0.000019,0.726127,-0.014049,0.818664,20.723400,0.037106,0.000009,0.055614,7.877810e-07,35651585.0,1.039530,4.0,1.47531,NaN,9.757030e-07,0.0,0.229726,105.864119,0.000017,3.792412e+15,NaN,

In [41]:
# Rank top anomalies
df_ranked_med = df_anom.sort_values("dimmad_medmed", ascending=True).copy()
df_ranked_med[["ZTFID","dimmad_medmed"]].head(20)


,ZTFID,dimmad_medmed
460,ZTF20abyrxie,-0.756196
12601,ZTF24abrirtr,-0.574733
5660,ZTF22aahfzfd,-0.506736
5826,ZTF22aajbpuo,-0.489937
13088,ZTF25aadnvfe,-0.481751
5616,ZTF22aagwane,-0.443242
4340,ZTF21abwntdi,-0.431223
9296,ZTF23aaqrlwy,-0.427159
5835,ZTF22aajfxrx,-0.424419
12153,ZTF24abgoksi,-0.422766


In [210]:
df_out = (
    df_anom[["ZTFID", "dimmad_medmed"]]
    .sort_values("dimmad_medmed", ascending=True)
    .reset_index(drop=True)
)

df_out["rank"] = df_out.index + 1

rank_path = OUT_DIR / "relaissranked_medmed_.csv"
df_out[["ZTFID", "rank"]].to_csv(rank_path, index=False)

rank_path

PosixPath('outputs/relaissranked_medmed_.csv')

In [211]:
from pathlib import Path
from joblib import dump

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

bundle_path = MODEL_DIR / "dimmad_medmed_relaiss_bundle.joblib"

dump(
    {
        "feature_cols": matchfeature_cols,   # the RELAISS-only feature list
        "imputer": imputer,                  # fitted
        "scaler": scaler,                    # fitted
        "model": dimmad_medmed,              # trained DiMMAD (medmed)
    },
    bundle_path,
)

bundle_path

PosixPath('models/dimmad_medmed_relaiss_bundle.joblib')

## Compare aggregation choices (min-median)

In [59]:
dimmad_minmed = DistanceAnomaly(
    cluster_agg="min",
    metric_agg="median",
    normalize_scores=True
)

dimmad_minmed.fit(X_train, y_train)
scores_minmed = dimmad_minmed.score_samples(X_score)

df_anom["dimmad_minmed"] = scores_minmed
df_anom[["dimmad_medmed","dimmad_minmed"]].head()


,dimmad_medmed,dimmad_minmed
0,-4.187967e-12,-0.001503
1,-2.158663e-14,-0.001324
2,-1.202602e-13,-0.001224
3,-6.721686e-14,-0.001417
4,-5.004757e-14,-0.000662


In [60]:
df_ranked_min = df_anom.sort_values("dimmad_minmed", ascending=True).copy()
df_ranked_min[["ZTFID","dimmad_minmed"]].head(20)


,ZTFID,dimmad_minmed
11570,ZTF24aaemydm,-0.975135
6342,ZTF22aakdbia,-0.935496
9370,ZTF23aaixyan,-0.692445
6263,ZTF22aajfxrx,-0.272871
9139,ZTF23aaflnxp,-0.263776
6760,ZTF22aaoonxc,-0.230819
6256,ZTF22aajbqel,-0.210001
6293,ZTF22aajkifc,-0.203057
6255,ZTF22aajbpzc,-0.194996
6426,ZTF22aakmpud,-0.148340


## Fetch lightcurves / cutouts from ALeRCE



In [43]:
from joblib import load

# Load DiMMAD bundle
DIMMAD_BUNDLE_PATH = "models/dimmad_medmed_relaiss_bundle.joblib"
art_dm = load(DIMMAD_BUNDLE_PATH)

dimmad_val       = art_dm["model"]
knn_imp_dm       = art_dm["imputer"]
scaler_dm        = art_dm["scaler"]
dm_feature_cols  = art_dm["feature_cols"]

In [91]:
from alerce.core import Alerce
from tqdm import tqdm
alerce = Alerce()

oids = objs_top["oid"].tolist()
print(f"Fetched {len(oids)} transient candidates from ALeRCE")

DIMMAD_BUNDLE_PATH = "models/dimmad_medmed_relaiss_bundle.joblib"
art_dm = load(DIMMAD_BUNDLE_PATH)

dimmad_val       = art_dm["model"]
knn_imp_dm       = art_dm["imputer"]
scaler_dm        = art_dm["scaler"]
dm_feature_cols  = art_dm["feature_cols"]

# Fetch up to 1000 recent objects classified as Transient by ALeRCE lc_classifier
objs_top = alerce.query_objects(
    classifier="lc_classifier_top",
    class_name="Transient",
    probability=0.7,
    page_size=1000,
    order_by="lastmjd",
    order_mode="DESC",
    format="pandas"
).drop_duplicates("oid")

oids = objs_top["oid"].tolist()
print(f"Fetched {len(oids)} transient candidates from ALeRCE")


Fetched 1000 transient candidates from ALeRCE
Fetched 1000 transient candidates from ALeRCE


In [93]:
# Cross-check with stamp classifier to keep high-confidence SNe
probs_list = []
for oid in tqdm(oids, desc="Fetching probabilities"):
    try:
        p = alerce.query_probabilities(oid, format="pandas", survey="ztf")
        if p is not None and not p.empty:
            p["oid"] = oid
            probs_list.append(p)
    except Exception as e:
        print(f"Failed for {oid}: {e}")

probs_stamp = pd.concat(probs_list, ignore_index=True) if probs_list else pd.DataFrame()

wide = probs_stamp.pivot_table(
    index="oid", columns="class_name", values="probability", aggfunc="max"
).fillna(0.0)

if "SN" not in wide.columns:
    wide["SN"] = 0.0

cand    = wide[wide["SN"] >= 0.7].index
sn_objs = objs_top[objs_top["oid"].isin(cand)].copy()
oids    = sn_objs["oid"].tolist()

print(f"Total Transients: {len(objs_top)}  |  Likely SNe (stamp P ≥ 0.7): {len(sn_objs)}")
sn_objs.head()


Fetching probabilities: 100%|███████████████| 1000/1000 [14:44<00:00,  1.13it/s]


Total Transients: 1000  |  Likely SNe (stamp P ≥ 0.7): 185


,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,g_r_mean,g_r_mean_corr,firstmjd,lastmjd,deltajd,meanra,meandec,sigmara,sigmadec,class,classifier,probability,step_id_corr
7,ZTF18admikzo,1032,2385,58261.482188,61125.518704,True,False,263,None,None,None,None,59733.354375,61125.518704,1392.164329,287.130284,-0.415086,0.004706,0.004706,Transient,lc_classifier_top,0.824,27.5.7a32.dev1
10,ZTF19addazgo,114,1906,58642.274525,61125.501586,False,False,36,None,None,None,None,59806.174028,61125.501586,1319.327558,265.153586,-4.061844,0.012201,0.012170,Transient,lc_classifier_top,0.994,27.5.7a32.dev1
12,ZTF18aaxxfah,110,1450,58258.251586,61125.486250,False,False,31,None,None,None,None,58273.315718,61125.486250,2852.170532,223.701664,5.507450,0.012603,0.012545,Transient,lc_classifier_top,0.904,27.5.7a32.dev1
14,ZTF18abltdkd,203,4041,58249.330359,61125.468218,True,False,26,None,None,None,None,58337.183438,61125.468218,2788.284780,243.586172,3.231931,0.002751,0.002746,Transient,lc_classifier_top,0.910,27.5.7a32.dev1
15,ZTF19aavhxby,305,3045,58243.335359,61125.464919,False,False,81,None,None,None,None,58628.313229,61125.464919,2497.151690,224.137059,-2.106456,0.002174,0.002173,Transient,lc_classifier_top,1.000,27.5.7a32.dev1


In [63]:
probs_list = []
for oid in tqdm(oids, desc="Fetching probabilities"):
    try:
        p = alerce.query_probabilities(oid, format="pandas", survey="ztf")
        if p is not None and not p.empty:
            p["oid"] = oid
            probs_list.append(p)
    except Exception as e:
        print(f"Failed for {oid}: {e}")


Fetching probabilities: 100%|███████████████| 1000/1000 [14:18<00:00,  1.16it/s]


In [69]:
probs_stamp = pd.concat(probs_list, ignore_index=True) if probs_list else pd.DataFrame()
wide = probs_stamp.pivot_table(
    index="oid", columns="class_name", values="probability", aggfunc="max"
).fillna(0.0)
if "SN" not in wide.columns:
    wide["SN"] = 0.0

cand    = wide[wide["SN"] >= 0.7].index
sn_objs = objs_top[objs_top["oid"].isin(cand)].copy()
oids    = sn_objs["oid"].tolist()
print(f"Likely SNe (stamp P >= 0.7): {len(sn_objs)}")

Likely SNe (stamp P >= 0.7): 185


In [75]:
# Features
rows = []
for oid in tqdm(oids, desc="Fetching features"):
    try:
        f = alerce.query_features(oid, format="pandas")
        if f is not None and not f.empty:
            fw = f.pivot_table(
                index="oid", columns="name", values="value", aggfunc="first"
            )
            rows.append(fw)
        else:
            rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))
    except:
        rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))

df_new = pd.concat(rows, axis=0) if rows else pd.DataFrame()
print(f"Features fetched for {len(df_new)} objects")


Fetching features: 100%|██████████████████████| 185/185 [03:01<00:00,  1.02it/s]

Features fetched for 185 objects


In [77]:
#  Load DiMMAD bundle and score
art_dm          = load("models/dimmad_medmed_relaiss_bundle.joblib")
dimmad_val      = art_dm["model"]
knn_imp_dm      = art_dm["imputer"]
scaler_dm       = art_dm["scaler"]
dm_feature_cols = art_dm["feature_cols"]

# Align columns
for c in dm_feature_cols:
    if c not in df_new.columns:
        df_new[c] = np.nan

X_dm_new    = df_new[dm_feature_cols].replace([np.inf, -np.inf], np.nan)
X_dm_imp    = knn_imp_dm.transform(X_dm_new)
X_dm_scaled = scaler_dm.transform(X_dm_imp)
scores_dm   = dimmad_val.score_samples(X_dm_scaled)


In [79]:
# Step 4: Rank and display results
res_dm = pd.DataFrame({
    "oid":          df_new.index,
    "dimmad_score": scores_dm,
}).reset_index(drop=True)

res_dm["dimmad_rank"] = res_dm["dimmad_score"].rank(
    method="first", ascending=False
).astype(int)

keep_cols = [c for c in ["oid", "meanra", "meandec", "ndet", "firstmjd", "lastmjd"]
             if c in sn_objs.columns]
res_dm = res_dm.merge(sn_objs[keep_cols], on="oid", how="left")

print(f"\nDiMMAD ALeRCE validation — {len(res_dm)} objects scored")
print(f"Score range: {scores_dm.min():.4f} to {scores_dm.max():.4f}")
print(f"Top 10 anomaly candidates:")
print(res_dm.sort_values("dimmad_rank").head(10))


DiMMAD ALeRCE validation — 185 objects scored
Score range: -0.0000 to -0.0000
Top 10 anomaly candidates:
            oid  dimmad_score  dimmad_rank      meanra    meandec  ndet  \
0  ZTF18admikzo          -0.0            1  287.130284  -0.415086   263   
1  ZTF19addazgo          -0.0            2  265.153586  -4.061844    36   
2  ZTF18aaxxfah          -0.0            3  223.701664   5.507450    31   
3  ZTF18abltdkd          -0.0            4  243.586172   3.231931    26   
4  ZTF19aavhxby          -0.0            5  224.137059  -2.106456    81   
5  ZTF19aawkakl          -0.0            6  272.318841   8.220046    38   
6  ZTF20actnzzk          -0.0            7  186.415800  -2.949805    68   
7  ZTF20actnzxm          -0.0            8  184.467619  -0.908026    81   
8  ZTF20acuosvy          -0.0            9  187.209440  -1.944074    57   
9  ZTF20aaawwkl          -0.0           10  152.560818  23.108912   612   

       firstmjd       lastmjd  
0  59733.354375  61125.518704  
1  5

In [81]:
# Check feature matrix before and after preprocessing
print("=== Raw features ===")
print(f"All NaN: {X_dm_new.isna().all(axis=1).sum()} rows")
print(f"Any values: {X_dm_new.notna().any(axis=1).sum()} rows")
print(X_dm_new.iloc[0])

print("\n=== After imputation ===")
print(f"Any non-zero: {(X_dm_imp != 0).any()}")
print(f"Unique values sample: {np.unique(X_dm_imp.flatten())[:10]}")

print("\n=== After scaling ===")
print(f"Any non-zero: {(X_dm_scaled != 0).any()}")
print(f"Unique values sample: {np.unique(X_dm_scaled.flatten())[:10]}")

print("\n=== Raw scores ===")
print(f"Unique scores: {np.unique(scores_dm)}")
print(f"Score dtype: {scores_dm.dtype}")

print("\n=== Feature column match ===")
alerce_cols = set(df_new.columns)
dm_cols = set(dm_feature_cols)
print(f"DiMMAD expects {len(dm_feature_cols)} features")
print(f"Found in ALeRCE data: {len(alerce_cols & dm_cols)}")
print(f"Missing from ALeRCE: {dm_cols - alerce_cols}")

=== Raw features ===
All NaN: 185 rows
Any values: 0 rows
g_peak_time                  NaN
r_peak_time                  NaN
g_rise_time                  NaN
g_decline_time               NaN
r_rise_time                  NaN
r_decline_time               NaN
g_duration_above_half_flux   NaN
r_duration_above_half_flux   NaN
g_amplitude                  NaN
r_amplitude                  NaN
g_skewness                   NaN
r_skewness                   NaN
g_beyond_2sigma              NaN
r_beyond_2sigma              NaN
mean_g-r                     NaN
g-r_at_g_peak                NaN
mean_color_rate              NaN
g_max_rolling_variance       NaN
r_max_rolling_variance       NaN
g_mean_rolling_variance      NaN
r_mean_rolling_variance      NaN
g_rise_local_curvature       NaN
g_decline_local_curvature    NaN
r_rise_local_curvature       NaN
r_decline_local_curvature    NaN
Name: ZTF18admikzo, dtype: float64

=== After imputation ===
Any non-zero: True
Unique values sample: [-1.59157981e+0

In [83]:
print(dm_feature_cols[:10])

['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude']


In [89]:
# Make sure you're using the filtered oids
print(f"sn_objs count: {len(sn_objs)}")
print(f"oids count: {len(oids)}")

# Re-fetch using only the filtered SN oids
oids_sn = sn_objs["oid"].tolist()
df_new, bad = fetch_features(oids_sn, keep_placeholders=True)
print(f"Non-empty rows: {df_new.notna().any(axis=1).sum()}")
print(df_new.iloc[0])

sn_objs count: 185
oids count: 185


NameError: name 'fetch_features' is not defined